In [ ]:
# Kaggle Playground Series S6E4 — Predicting Irrigation Need
# Strateji: Grandmaster Stacking
#   Katman 1: LightGBM + XGBoost + CatBoost  (5-fold CV)
#   Katman 2: Logistic Regression meta-model
# Metric  : Balanced Accuracy
# Gereksinimler:
#   pip install lightgbm xgboost catboost scikit-learn pandas numpy

# CELL 1: Imports
import gc, os, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

SEED     = 42
N_FOLDS  = 5
TARGET   = "Irrigation_Need"
ID_COL   = "id"

# CELL 2: Veri Yükleme
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

print(f"Train : {train.shape}")
print(f"Test  : {test.shape}")
print(f"\nTarget:\n{train[TARGET].value_counts()}\n")

le = LabelEncoder()
y  = le.fit_transform(train[TARGET])
N_CLASSES = len(le.classes_)
print(f"Sınıflar: {list(le.classes_)} → {list(range(N_CLASSES))}")

# CELL 3: Orijinal Veri Ekle
for fname in ["original.csv", "original_data.csv"]:
    if os.path.exists(fname):
        orig = pd.read_csv(fname)
        for alt in ["irrigation_need", "label", "Class", "Irrigation Need"]:
            if alt in orig.columns:
                orig = orig.rename(columns={alt: TARGET})
                break
        if TARGET not in orig.columns:
            break
        orig   = orig.drop_duplicates().reset_index(drop=True)
        shared = [c for c in train.columns if c in orig.columns and c != ID_COL]
        orig_sub = orig[shared].copy()
        orig_sub[ID_COL] = -1
        n0    = len(train)
        train = pd.concat([train, orig_sub], ignore_index=True)
        y     = le.transform(train[TARGET])
        print(f"✅ {fname} eklendi: +{len(train)-n0} satır → Toplam: {len(train)}")
        break
else:
    print("⚠️  original.csv bulunamadı")

# CELL 4: Feature Engineering
def engineer(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    c  = df.columns.tolist()

    # Magic binary eşikler (Chris Deotte, rank-1)
    if "Soil_Moisture"  in c: df["soil_lt_25"]  = (df["Soil_Moisture"]  < 25).astype("int8")
    if "Temperature_C"  in c: df["temp_gt_30"]  = (df["Temperature_C"]  > 30).astype("int8")
    if "Rainfall_mm"    in c: df["rain_lt_300"] = (df["Rainfall_mm"]    < 300).astype("int8")
    if "Wind_Speed_kmh" in c: df["wind_gt_10"]  = (df["Wind_Speed_kmh"] > 10).astype("int8")
    if "Soil_Moisture"  in c:
        df["soil_lt_20"] = (df["Soil_Moisture"] < 20).astype("int8")
        df["soil_lt_30"] = (df["Soil_Moisture"] < 30).astype("int8")
    if "Temperature_C"  in c:
        df["temp_gt_25"] = (df["Temperature_C"] > 25).astype("int8")
        df["temp_gt_35"] = (df["Temperature_C"] > 35).astype("int8")
    if "Rainfall_mm"    in c:
        df["rain_lt_200"] = (df["Rainfall_mm"] < 200).astype("int8")
        df["rain_lt_400"] = (df["Rainfall_mm"] < 400).astype("int8")
    if "Wind_Speed_kmh" in c:
        df["wind_gt_15"] = (df["Wind_Speed_kmh"] > 15).astype("int8")
    if "Humidity"       in c:
        df["humid_lt_40"] = (df["Humidity"] < 40).astype("int8")
        df["humid_gt_70"] = (df["Humidity"] > 70).astype("int8")

    # Formül skoru
    high = pd.Series(0.0, index=df.index)
    low  = pd.Series(0.0, index=df.index)
    if "Soil_Moisture"  in c: high += (df["Soil_Moisture"]  < 25)  * 2
    if "Rainfall_mm"    in c: high += (df["Rainfall_mm"]    < 300) * 2
    if "Temperature_C"  in c: high += (df["Temperature_C"]  > 30)  * 1
    if "Wind_Speed_kmh" in c: high += (df["Wind_Speed_kmh"] > 10)  * 1
    if "Crop_Growth_Stage" in c:
        low += (df["Crop_Growth_Stage"] == "Harvest") * 2
        low += (df["Crop_Growth_Stage"] == "Sowing")  * 2
    if "Mulching_Used" in c:
        low += df["Mulching_Used"].astype(str).str.lower().isin(["yes","true","1"]).astype(int)
    df["formula_score"] = (high - low).astype("int8")

    # Fiziksel etkileşimler
    if all(x in c for x in ["Temperature_C","Wind_Speed_kmh","Humidity"]):
        df["ET_proxy"] = (df["Temperature_C"] * df["Wind_Speed_kmh"]) / (df["Humidity"] + 1)
    if all(x in c for x in ["Soil_Moisture","Rainfall_mm"]):
        df["water_deficit"] = df["Soil_Moisture"] - df["Rainfall_mm"] * 0.1
    if all(x in c for x in ["Evapotranspiration","Rainfall_mm"]):
        df["net_water"] = df["Evapotranspiration"] - df["Rainfall_mm"] * 0.7

    # 2-yönlü kategorik kombinasyonlar
    cat_pairs = [
        ("Crop_Growth_Stage","Mulching_Used"),
        ("Crop_Type","Crop_Growth_Stage"),
        ("Crop_Type","Season"),
        ("Soil_Type","Season"),
        ("Irrigation_Method","Crop_Growth_Stage"),
    ]
    for a, b in cat_pairs:
        if a in df.columns and b in df.columns:
            df[f"{a}_x_{b}"] = pd.factorize(
                df[a].astype(str) + "||" + df[b].astype(str)
            )[0].astype("int16")

    for col in df.select_dtypes("bool").columns:
        df[col] = df[col].astype("int8")
    return df

train = engineer(train)
test  = engineer(test)
gc.collect()
print(f"Feature engineering tamam → Train: {train.shape} | Test: {test.shape}")

# CELL 5: Feature Listesi & Encoding
FEAT_COLS = [c for c in train.columns if c not in [ID_COL, TARGET]]

STR_COLS  = [c for c in FEAT_COLS if train[c].dtype == "object"]
NUM_COLS  = [c for c in FEAT_COLS if c not in STR_COLS]

# CatBoost için: string fill
X_cb      = train[FEAT_COLS].copy()
X_cb_test = test[FEAT_COLS].copy()
for df in [X_cb, X_cb_test]:
    df[STR_COLS] = df[STR_COLS].fillna("MISSING").astype(str)
    df[NUM_COLS] = df[NUM_COLS].fillna(df[NUM_COLS].median()).astype("float32")
cat_idx = [X_cb.columns.get_loc(c) for c in STR_COLS]

# LGB / XGB için: OrdinalEncoder
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_num      = train[FEAT_COLS].copy()
X_num_test = test[FEAT_COLS].copy()
for df in [X_num, X_num_test]:
    df[STR_COLS] = df[STR_COLS].fillna("MISSING").astype(str)
    df[NUM_COLS] = df[NUM_COLS].fillna(df[NUM_COLS].median()).astype("float32")
X_num[STR_COLS]      = oe.fit_transform(X_num[STR_COLS]).astype("float32")
X_num_test[STR_COLS] = oe.transform(X_num_test[STR_COLS]).astype("float32")
X_num      = X_num.astype("float32")
X_num_test = X_num_test.astype("float32")

print(f"Feature sayısı: {len(FEAT_COLS)} | Kategorik: {len(STR_COLS)}")

# ── CELL 6: Ortak CV Yapısı ───────────────────────────────────────
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Stacking OOF ve test tahminleri (her model için)
lgb_oof   = np.zeros((len(train), N_CLASSES))
lgb_test  = np.zeros((len(X_num_test), N_CLASSES))

xgb_oof   = np.zeros((len(train), N_CLASSES))
xgb_test  = np.zeros((len(X_num_test), N_CLASSES))

cb_oof    = np.zeros((len(train), N_CLASSES))
cb_test   = np.zeros((len(X_cb_test), N_CLASSES))

# CELL 7a: LightGBM
print("\n" + "="*50)
print("KATMAN 1-A: LightGBM")
print("="*50)

LGB_PARAMS = dict(
    objective        = "multiclass",
    num_class        = N_CLASSES,
    num_leaves       = 63,
    max_depth        = 6,
    learning_rate    = 0.05,
    n_estimators     = 3000,
    min_child_samples= 20,
    feature_fraction = 0.8,
    bagging_fraction = 0.8,
    bagging_freq     = 5,
    lambda_l1        = 0.1,
    lambda_l2        = 0.1,
    class_weight     = "balanced",
    random_state     = SEED,
    n_jobs           = -1,
    verbosity        = -1,
)

lgb_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_num, y)):
    X_tr, y_tr = X_num.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X_num.iloc[va_idx],  y[va_idx]

    m = lgb.LGBMClassifier(**LGB_PARAMS)
    m.fit(X_tr, y_tr,
          eval_set=[(X_va, y_va)],
          callbacks=[lgb.early_stopping(150, verbose=False),
                     lgb.log_evaluation(-1)])

    lgb_oof[va_idx] = m.predict_proba(X_va)
    lgb_test        += m.predict_proba(X_num_test) / N_FOLDS

    score = balanced_accuracy_score(y_va, np.argmax(lgb_oof[va_idx], axis=1))
    lgb_scores.append(score)
    print(f"  Fold {fold+1}/{N_FOLDS} | BA = {score:.5f} | iter = {m.best_iteration_}")
    del m; gc.collect()

lgb_oof_ba = balanced_accuracy_score(y, np.argmax(lgb_oof, axis=1))
print(f"  LGB OOF BA: {lgb_oof_ba:.5f}")

# CELL 7b: XGBoost
print("\n" + "="*50)
print("KATMAN 1-B: XGBoost")
print("="*50)

XGB_PARAMS = dict(
    objective        = "multi:softprob",
    num_class        = N_CLASSES,
    n_estimators     = 3000,
    learning_rate    = 0.05,
    max_depth        = 6,
    min_child_weight = 5,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_alpha        = 0.1,
    reg_lambda       = 1.0,
    tree_method      = "hist",
    eval_metric      = "mlogloss",
    random_state     = SEED,
    n_jobs           = -1,
)

from sklearn.utils.class_weight import compute_sample_weight
xgb_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_num, y)):
    X_tr, y_tr = X_num.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X_num.iloc[va_idx],  y[va_idx]
    sw = compute_sample_weight("balanced", y_tr)

    m = xgb.XGBClassifier(**XGB_PARAMS)
    m.fit(X_tr, y_tr,
          sample_weight=sw,
          eval_set=[(X_va, y_va)],
          early_stopping_rounds=150,
          verbose=False)

    xgb_oof[va_idx] = m.predict_proba(X_va)
    xgb_test        += m.predict_proba(X_num_test) / N_FOLDS

    score = balanced_accuracy_score(y_va, np.argmax(xgb_oof[va_idx], axis=1))
    xgb_scores.append(score)
    print(f"  Fold {fold+1}/{N_FOLDS} | BA = {score:.5f} | iter = {m.best_iteration}")
    del m; gc.collect()

xgb_oof_ba = balanced_accuracy_score(y, np.argmax(xgb_oof, axis=1))
print(f"  XGB OOF BA: {xgb_oof_ba:.5f}")

# CELL 7c: CatBoost
print("\n" + "="*50)
print("KATMAN 1-C: CatBoost")
print("="*50)

CB_PARAMS = dict(
    iterations        = 3000,
    learning_rate     = 0.05,
    depth             = 6,
    l2_leaf_reg       = 3,
    bootstrap_type    = "Bernoulli",
    subsample         = 0.8,
    colsample_bylevel = 0.8,
    auto_class_weights= "Balanced",
    loss_function     = "MultiClass",
    eval_metric       = "TotalF1",
    random_seed       = SEED,
    task_type         = "CPU",
    thread_count      = -1,
    verbose           = False,
)

cb_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y)):
    X_tr, y_tr = X_cb.iloc[tr_idx], y[tr_idx]
    X_va, y_va = X_cb.iloc[va_idx],  y[va_idx]

    tr_pool  = Pool(X_tr,   y_tr, cat_features=cat_idx)
    val_pool = Pool(X_va,   y_va, cat_features=cat_idx)
    tst_pool = Pool(X_cb_test,    cat_features=cat_idx)

    m = CatBoostClassifier(**CB_PARAMS)
    m.fit(tr_pool, eval_set=val_pool, early_stopping_rounds=200)

    cb_oof[va_idx] = m.predict_proba(val_pool)
    cb_test        += m.predict_proba(tst_pool) / N_FOLDS

    score = balanced_accuracy_score(y_va, np.argmax(cb_oof[va_idx], axis=1))
    cb_scores.append(score)
    print(f"  Fold {fold+1}/{N_FOLDS} | BA = {score:.5f} | iter = {m.best_iteration_}")
    del tr_pool, val_pool, tst_pool, m; gc.collect()

cb_oof_ba = balanced_accuracy_score(y, np.argmax(cb_oof, axis=1))
print(f"  CB OOF BA: {cb_oof_ba:.5f}")

# CELL 8: Stacking — Meta Model
print("\n" + "="*50)
print("KATMAN 2: Logistic Regression Meta-Model")
print("="*50)

# OOF olasılıklarını meta-feature olarak birleştir
meta_train = np.hstack([lgb_oof, xgb_oof, cb_oof])          # shape: (N, 9)
meta_test  = np.hstack([lgb_test, xgb_test, cb_test])        # shape: (M, 9)

# 5-fold CV ile meta-model
meta_oof   = np.zeros((len(meta_train), N_CLASSES))
meta_test_preds = np.zeros((len(meta_test), N_CLASSES))
meta_scores = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(meta_train, y)):
    X_tr_m, y_tr_m = meta_train[tr_idx], y[tr_idx]
    X_va_m, y_va_m = meta_train[va_idx],  y[va_idx]

    meta_model = LogisticRegression(
        multi_class  = "multinomial",
        solver       = "lbfgs",
        max_iter     = 1000,
        class_weight = "balanced",
        C            = 1.0,
        random_state = SEED,
    )
    meta_model.fit(X_tr_m, y_tr_m)

    meta_oof[va_idx]     = meta_model.predict_proba(X_va_m)
    meta_test_preds     += meta_model.predict_proba(meta_test) / N_FOLDS

    score = balanced_accuracy_score(y_va_m, np.argmax(meta_oof[va_idx], axis=1))
    meta_scores.append(score)
    print(f"  Fold {fold+1}/{N_FOLDS} | Meta BA = {score:.5f}")

meta_oof_ba = balanced_accuracy_score(y, np.argmax(meta_oof, axis=1))
print(f"\n  Meta OOF BA: {meta_oof_ba:.5f}")

# CELL 9: Özet
print("\n" + "="*50)
print("SONUÇLAR")
print("="*50)
print(f"  LightGBM  OOF BA : {lgb_oof_ba:.5f}")
print(f"  XGBoost   OOF BA : {xgb_oof_ba:.5f}")
print(f"  CatBoost  OOF BA : {cb_oof_ba:.5f}")
print(f"  Meta (Stack) BA  : {meta_oof_ba:.5f}  ← En iyi")

# CELL 10: Submission
final_preds  = le.inverse_transform(np.argmax(meta_test_preds, axis=1))
submission   = pd.DataFrame({"id": test[ID_COL], "Irrigation_Need": final_preds})
submission.to_csv("submission_grandmaster_stacking.csv", index=False)

print("\n" + "="*50)
print("✅ submission_grandmaster_stacking.csv kaydedildi!")
print(f"\nTahmin dağılımı:\n{pd.Series(final_preds).value_counts()}")
print("="*50)
print(f"\n🎯 Beklenen LB ≈ {meta_oof_ba:.5f}")

Train : (630000, 21)
Test  : (270000, 20)

Target:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Sınıflar: ['High', 'Low', 'Medium'] → [0, 1, 2]
✅ original.csv eklendi: +10000 satır → Toplam: 640000
Feature engineering tamam → Train: (640000, 41) | Test: (270000, 40)
Feature sayısı: 39 | Kategorik: 8

KATMAN 1-A: LightGBM
  Fold 1/5 | BA = 0.96834 | iter = 1198
  Fold 2/5 | BA = 0.96874 | iter = 1073
  Fold 3/5 | BA = 0.96765 | iter = 1115
  Fold 4/5 | BA = 0.96781 | iter = 1100
  Fold 5/5 | BA = 0.96419 | iter = 1063
  LGB OOF BA: 0.96735

KATMAN 1-B: XGBoost
  Fold 1/5 | BA = 0.96947 | iter = 1655
  Fold 2/5 | BA = 0.96892 | iter = 1689
  Fold 3/5 | BA = 0.96827 | iter = 1586
  Fold 4/5 | BA = 0.96852 | iter = 1589
  Fold 5/5 | BA = 0.96488 | iter = 1516
  XGB OOF BA: 0.96801

KATMAN 1-C: CatBoost
  Fold 1/5 | BA = 0.97070 | iter = 576
  Fold 2/5 | BA = 0.97107 | iter = 761
  Fold 3/5 | BA = 0.97147 | iter = 1043
  Fold 4/5 | BA = 0.971